# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fe4rlessCloak/flyrank-ml-workspace/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I am choosing Lane 4 because it answers a concrete question: which pages are getting seen but not clicked? My approach will be to predict expected CTR from page features (position, content type, intent, word count), then find pages where actual CTR falls below expected. 

The gap identifies pages where an editor can intervene by improving the title, meta description, or content to capture more clicks from existing visibility.



In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The decision:** Which page should an editor review to improve click-through performance?

**Who acts:** An SEO editor or content manager with limited weekly capacity. They are presented with a ranked list of pages where actual CTR falls below what the page's features suggest it should achieve.

**The action:** Review the top-K pages, inspect the reason codes (low CTR relative to position, weak engagement for the traffic level, content-length mismatch), and decide whether to rewrite the title or meta description, improve content depth, adjust the page to better match search intent, or monitor.

**Cost of a wrong call:**

*False positive (flagging a page that is performing as expected):* the editor wastes time reviewing a healthy page.

*False negative (missing a page with real CTR opportunity):* the page continues underperforming, leaving clicks and traffic on the table.

**Why ML helps:** CTR depends on many factors such as position, content type, intent, word count, content age, that interact in non-linear ways. A model can learn the expected CTR for each page given its specific profile, then flag only the pages that deviate meaningfully from that expectation.



In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*


Three numbers from the starter dataset that make Lane 4 worth the next 7 weeks:

**1. 85.2% of visible pages have CTR below 0.5%.** Of the 16,726 pages with at least 500 impressions in 90 days, 14,245 have a click-through rate under half a percent. These pages get seen but barely clicked, a massive pool of potential improvement.

**2. CTR varies 3x within the same position tier.** At position tier "page_1" (positions 4-10), the median CTR is 0.24%, but the top quarter of pages get 0.44% while the bottom quarter get just 0.12%. Position alone doesn't explain CTR, content factors like title, meta description, and intent match matter.

**3. 6,741 high-visibility pages leave clicks on the table.** These pages have 3,000+ impressions in 90 days yet CTR below 0.5%. Even a small improvement — say from 0.3% to 0.6% — would mean thousands of additional clicks. That's the opportunity this lane targets.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

visible = df[df['impressions_90d'] >= 500].copy()

# Number 1: How many visible pages have very low CTR?
low_ctr = visible[visible['ctr'] < 0.5]  
print(f"1. Visible pages (>=500 impressions): {len(visible):,}")
print(f"   Pages with CTR < 0.5%: {len(low_ctr):,} ({len(low_ctr)/len(visible):.1%})")
print(f"   → Thousands of pages get seen but barely clicked.\n")


# Number 2: CTR varies hugely at the same position
for tier in ['top_3', 'page_1', 'striking']:
    tier_pages = visible[visible['position_tier'] == tier]
    if len(tier_pages) > 0:
        median_ctr = tier_pages['ctr'].median()
        p25 = tier_pages['ctr'].quantile(0.25)
        p75 = tier_pages['ctr'].quantile(0.75)
        print(f"2. Position tier '{tier}': {len(tier_pages):,} pages")
        print(f"   Median CTR: {median_ctr:.2f}%  (25th-75th: {p25:.2f}% - {p75:.2f}%)")
        print(f"   → At the same position, some pages get 3x the CTR of others.\n")

# Number 3: How many pages have both high impressions AND low CTR?
high_impression_low_ctr = visible[(visible['impressions_90d'] >= 3000) & (visible['ctr'] < 0.5)]
print(f"3. Pages with >=3,000 impressions AND CTR < 0.5%: {len(high_impression_low_ctr):,}")
print(f"   → These are high-visibility pages leaving clicks on the table.")



1. Visible pages (>=500 impressions): 16,726
   Pages with CTR < 0.5%: 14,245 (85.2%)
   → Thousands of pages get seen but barely clicked.

2. Position tier 'top_3': 458 pages
   Median CTR: 0.20%  (25th-75th: 0.08% - 0.49%)
   → At the same position, some pages get 3x the CTR of others.

2. Position tier 'page_1': 7,064 pages
   Median CTR: 0.24%  (25th-75th: 0.12% - 0.44%)
   → At the same position, some pages get 3x the CTR of others.

2. Position tier 'striking': 4,485 pages
   Median CTR: 0.17%  (25th-75th: 0.08% - 0.34%)
   → At the same position, some pages get 3x the CTR of others.

3. Pages with >=3,000 impressions AND CTR < 0.5%: 6,741
   → These are high-visibility pages leaving clicks on the table.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**

- I can claim that certain observed signals (position, content type, intent, word count, content age) are associated with CTR in this dataset.
- I can claim that a model predicting expected CTR from these signals can identify pages where actual CTR falls below what similar pages achieve, measured by gap size on held-out data.
- I can claim directional recommendations: "this page's CTR is lower than 80% of similar pages at the same position and it may benefit from a title or meta description review."

**What I cannot claim:**

- I cannot claim that improving CTR will improve rankings as this data shows correlation, not causation.
- I cannot claim that a low CTR is caused by a specific factor (title, meta, content), I can only flag that a gap exists and suggest possible reasons.
- I cannot claim that I am predicting Google's algorithm, ranking factors, or AI citations.
- I cannot claim that fixing a page will definitely recover its CTR, only that it shows patterns worth a human review.
- I cannot claim that the warehouse-scale result will match the starter slice, the result must be earned again with proper validation.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Do pages with low CTR still have real traffic at stake?
low_ctr_high_volume = df[(df['ctr'] < 0.5) & (df['impressions_90d'] >= 1000)]
print(f"Pages with CTR < 0.5% and >= 1,000 impressions: {len(low_ctr_high_volume):,}")
print(f"Total potential impressions across these pages: {low_ctr_high_volume['impressions_90d'].sum():,}")
print(f"→ Even a small CTR improvement on these pages could drive meaningful additional clicks.")




Pages with CTR < 0.5% and >= 1,000 impressions: 11,379
Total potential impressions across these pages: 122,742,316
→ Even a small CTR improvement on these pages could drive meaningful additional clicks.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.